# The Nonhierarchical Bayesian Finite Element Method: FRP DIC test

In [ ]:
import os
import numpy as np
from scipy.sparse import diags_array
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import Circle, Rectangle
from matplotlib.cm import ScalarMappable
import seaborn as sns

from myjivex.util import QuickViewer, ElemViewer

from fem.jive import CJiveRunner
from probability.multivariate import Gaussian, SymbolicCovariance
from probability.process import GaussianProcess, ZeroMeanFunction, SquaredExponential
from util.linalg import Matrix

from experiments.reproduction.nonhierarchical.frp_dic.props import get_fem_props
from experiments.reproduction.nonhierarchical.frp_dic import caching, params, misc

In [ ]:
# matplotlib settings
plt.rc("text", usetex=True)
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Computer Modern Roman"]
plt.rcParams["font.size"] = 12
plt.rcParams["legend.fontsize"] = 10
plt.rcParams["text.latex.preamble"] = r"\usepackage{xfrac}"

## Forward problem

We first consider the general model setup

### Figure 8a: micromodel overview

In this figure, we show the full micromodel.
Fibers are indicated in gray.
The dashed line indicates the region where the DIC is employed.

In [ ]:
fibers = caching.get_or_calc_fibers()
grid = caching.get_or_calc_dic_grid()

rve_size = params.geometry_params["rve_size"]
obs_size = params.geometry_params["obs_size"]
r_fiber = params.geometry_params["r_fiber"]

fig, ax = plt.subplots(figsize=(4.8, 4.8))

for fiber in fibers:
    ax.add_patch(Circle(fiber, r_fiber, color="0.5", alpha=0.5))

obs_box = Rectangle(
    (-obs_size, -obs_size), 2 * obs_size, 2 * obs_size, fc="none", ec="k", ls="--"
)

ax.add_patch(obs_box)

ax.set_aspect("equal")
ax.set_xlim((-rve_size, rve_size))
ax.set_ylim((-rve_size, rve_size))
ax.set_xticks([-rve_size, 0.0, rve_size])
ax.set_yticks([-rve_size, 0.0, rve_size])
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
plt.show()

### Figure 8b: true stiffness field

In this figure, we show the full micromodel.
Fibers are indicated in gray.
The dashed line indicates the region where the DIC is employed.

In [ ]:
h = 0.050  # ground truth: 0.002
nodes, elems, egroups = caching.get_or_calc_mesh(h=h)

egroup = egroups["matrix"]
ipoints = caching.get_or_calc_ipoints(egroup=egroup, h=h)
distances = caching.get_or_calc_distances(egroup=egroup, h=h)
ip_stiffnesses = caching.get_or_calc_true_stiffnesses(egroup=egroup, h=h)

props = get_fem_props()
E_fiber = props["model"]["model"]["fiber"]["material"]["E"]
elem_stiffnesses = misc.calc_elem_stiffnesses(
    ip_stiffnesses, egroups, fiber_stiffness=np.inf
)

backdoor = {}
backdoor["xcoord"] = ipoints[:, 0]
backdoor["ycoord"] = ipoints[:, 1]
backdoor["e"] = ip_stiffnesses

jive = CJiveRunner(props, elems=elems, egroups=egroups)
globdat = jive(**backdoor)

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))

E_matrix = params.material_params["E_matrix"]
cs = ElemViewer(
    elem_stiffnesses,
    globdat,
    ax=ax,
    colormap=sns.cm.rocket,
    colorbar=False,
    mincolor=0.0,
    maxcolor=E_matrix,
)

ax.set_aspect("equal", adjustable="box")
ax.set_axis_on()
ax.set_xlim([-rve_size, rve_size])
ax.set_ylim([-rve_size, rve_size])
ax.set_xticks([-rve_size, 0.0, rve_size])
ax.set_yticks([-rve_size, 0.0, rve_size])
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")

ax_pos = ax.get_position()
lax, bax, wax, hax = ax_pos.x0, ax_pos.y0, ax_pos.width, ax_pos.height
cax_pos = [lax + 0.1 * wax, bax + hax + 0.1, 0.8 * wax, 0.04]
cax = fig.add_axes(cax_pos)
cticks = np.array([0.0, 0.5, 1.0]) * E_matrix
fig.colorbar(cs, cax=cax, orientation="horizontal", ticks=cticks, format="%.0f")

# fname = "stiffness_h-{:.3f}.png".format(h)
# fname = os.path.join("plots", fname)
# os.makedirs(os.path.dirname(fname), exist_ok=True)
# plt.savefig(fname, dpi=600, bbox_inches="tight")

plt.show()

# # horizontal displacement field
# QuickViewer(globdat["state0"], globdat, comp=0, colormap=sns.cm.rocket)

# # horizontal strain field
# eps_xx, eps_yy, gamma_xy = misc.calc_strains(globdat)
# assert np.allclose(eps_xx.T, eps_xx[:,0]) # check constant strain
# ElemViewer(eps_xx[:,0], globdat, colormap=sns.cm.rocket)

### Figure 8c: DIC measurements

In this figure, the measurements of the horizontal strain $\varepsilon_{xx}$ are shown.

In [ ]:
fibers = caching.get_or_calc_fibers()
grid = caching.get_or_calc_dic_grid()
truth = caching.get_or_calc_true_dic_observations(h=h)

rve_size = params.geometry_params["rve_size"]
obs_size = params.geometry_params["obs_size"]
r_fiber = params.geometry_params["r_fiber"]

fig, ax = plt.subplots(figsize=(4.8, 4.8))

eps_xx = truth[::3]
cmap = LinearSegmentedColormap.from_list(
    "rocket_r_truncated", sns.cm.rocket(np.linspace(1.0, 0.2, 256))
)
norm = Normalize(vmin=np.min(eps_xx), vmax=0.0)

for i, square in enumerate(grid):
    color = cmap(norm(eps_xx[i]))
    ax.add_patch(Rectangle(square[:2], square[2], square[3], fc=color, ec="none"))

for fiber in fibers:
    ax.add_patch(Circle(fiber, r_fiber, fc="none", ec="0.7"))

ax.set_aspect("equal")
ax.set_xlim((-obs_size, obs_size))
ax.set_ylim((-obs_size, obs_size))
ax.set_xticks([-obs_size, 0.0, obs_size])
ax.set_yticks([-obs_size, 0.0, obs_size])
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")

ax_pos = ax.get_position()
lax, bax, wax, hax = ax_pos.x0, ax_pos.y0, ax_pos.width, ax_pos.height
cax_pos = [lax + 0.1 * wax, bax + hax + 0.1, 0.8 * wax, 0.04]
cax = fig.add_axes(cax_pos)
cticks = [np.min(eps_xx), 0.0]
sm = ScalarMappable(norm=norm, cmap=cmap)
fig.colorbar(sm, cax=cax, orientation="horizontal", ticks=cticks, format="%.3f")

# fname = "measurements_h-{:.3f}.png".format(h)
# fname = os.path.join("plots", fname)
# os.makedirs(os.path.dirname(fname), exist_ok=True)
# plt.savefig(fname, dpi=600, bbox_inches="tight")

plt.show()

## Figure 9a: prior distribution

In [ ]:
s = np.linspace(0.0, 0.2, 101)

target = GaussianProcess(
    mean=ZeroMeanFunction(),
    cov=SquaredExponential(l=0.02, sigma=2.0),
)

U, d, _ = np.linalg.svd(target.calc_cov(s, s))

trunc = 10
eigenfuncs = U[:, :trunc]
eigenvalues = d[:trunc]

kl_cov = SymbolicCovariance(Matrix(diags_array(eigenvalues), name="D"))
kl_target = Gaussian(mean=None, cov=kl_cov)

rng = np.random.default_rng(0)

E_matrix = params.material_params["E_matrix"]
alpha = params.material_params["alpha"]
beta = params.material_params["beta"]
c = params.material_params["c"]
d = params.material_params["d"]

saturation = misc.saturation(s, alpha, beta, c)
true_damage = misc.damage(saturation, d) * 100
E = E_matrix * (1 - true_damage)

damage_samples = []

for i in range(20):
    sample = eigenfuncs @ kl_target.calc_sample(rng)
    damage_sample = misc.sigmoid(sample, 1.0, 0.0) * 100
    damage_samples.append(damage_sample)

damage_samples = np.array(damage_samples)

color = cmap(0.5)
opacity = 0.3

fig, ax = plt.subplots()
for i, damage_sample in enumerate(damage_samples):
    if i == 0:
        ax.plot(s, damage_sample, color=sns.cm.rocket(0.5), zorder=1)
    else:
        ax.plot(s, damage_sample, color="0.5", alpha=0.5)

ax.plot(s, true_damage, color="k", linestyle="--")
ax.set_xlim((0, 0.2))
ax.set_ylim((0, 100))
ax.set_xlabel(r"distance to fiber (mm)")
ax.set_ylabel(r"stiffness reduction (\%)")
ax.set_xticks([0.00, 0.05, 0.10, 0.15, 0.20])
ax.set_yticks([0, 25, 50, 75, 100])

fname = "prior-samples_1d.pdf"
fname = os.path.join("plots", fname)
os.makedirs(os.path.dirname(fname), exist_ok=True)
plt.savefig(fname, bbox_inches="tight")

plt.show()

## Figure 9b: prior stiffness sample

In [ ]:
nodes, elems, egroups = caching.get_or_calc_mesh(h=h)
egroup = egroups["matrix"]
ipoints = caching.get_or_calc_ipoints(egroup=egroup, h=h)
distances = caching.get_or_calc_distances(egroup=egroup, h=h)

backdoor = {}
backdoor["xcoord"] = ipoints[:, 0]
backdoor["ycoord"] = ipoints[:, 1]
backdoor["e"] = np.zeros(ipoints.shape[0])

sample_idx = 0
damage_sample = damage_samples[sample_idx]

for ip, ipoint in enumerate(ipoints):
    dist = distances[ip]
    idx_l = int(dist / np.max(s) * (len(s) - 1))
    idx_r = idx_l + 1

    x_l = s[idx_l]
    x_r = s[idx_r]
    d_l = damage_sample[idx_l] / 100
    d_r = damage_sample[idx_r] / 100

    assert x_l <= dist <= x_r

    dam = d_l + (dist - x_l) / (x_r - x_l) * (d_r - d_l)
    backdoor["e"][ip] = E_matrix * (1 - dam)

elem_stiffness = np.zeros(len(elems))

for group_name, egroup in egroups.items():
    if group_name == "matrix":
        for ie, ielem in enumerate(egroup):
            ip_stiffness = backdoor["e"][3 * ie : 3 * (ie + 1)]
            elem_stiffness[ielem] = np.mean(ip_stiffness)
    elif group_name == "fiber":
        ielems = egroup.get_indices()
        elem_stiffness[ielems] = np.inf
    else:
        assert False

props = get_fem_props()
jive = CJiveRunner(props, elems=elems, egroups=egroups)
globdat = jive(**backdoor)

fig, ax = plt.subplots(figsize=(3, 3))

E_matrix = params.material_params["E_matrix"]
cs = ElemViewer(
    elem_stiffness,
    globdat,
    ax=ax,
    colormap=sns.cm.rocket,
    colorbar=False,
    mincolor=0.0,
    maxcolor=E_matrix,
)

ax.set_aspect("equal", adjustable="box")
ax.set_axis_on()
ax.set_xlim([-rve_size, rve_size])
ax.set_ylim([-rve_size, rve_size])
ax.set_xticks([-rve_size, 0.0, rve_size])
ax.set_yticks([-rve_size, 0.0, rve_size])
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")

ax_pos = ax.get_position()
lax, bax, wax, hax = ax_pos.x0, ax_pos.y0, ax_pos.width, ax_pos.height
cax_pos = [lax + 0.1 * wax, bax + hax + 0.1, 0.8 * wax, 0.04]
cax = fig.add_axes(cax_pos)
cticks = np.array([0.0, 0.5, 1.0]) * E_matrix
fig.colorbar(cs, cax=cax, orientation="horizontal", ticks=cticks, format="%.0f")

# fname = "stiffness_sample-{}_h-{:.3f}.png".format(sample_idx, h)
# fname = os.path.join("plots", fname)
# os.makedirs(os.path.dirname(fname), exist_ok=True)
# plt.savefig(fname, dpi=600, bbox_inches="tight")

plt.show()

# # horizontal displacement field
# QuickViewer(globdat["state0"], globdat, comp=0, colormap=sns.cm.rocket)

# # horizontal strain field
# eps_xx, eps_yy, gamma_xy = misc.calc_strains(globdat)
# assert np.allclose(eps_xx.T, eps_xx[:,0]) # check constant strain
# ElemViewer(eps_xx[:,0], globdat, colormap=sns.cm.rocket)